In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 24
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
16.04736780513803

Trial 1 =========================================
14.37420797422643

Trial 2 =========================================
13.256687963205826

Trial 3 =========================================
17.93756973629349

Trial 4 =========================================
13.628919188978331

Trial 5 =========================================
13.651794540164985

Trial 6 =========================================
13.57460867368526

Trial 7 =========================================
16.038602500030457

Trial 8 =========================================
14.424137611898903

Trial 9 =========================================
14.148899262046438

Trial 10 =========================================
14.574528377016225

Trial 11 =========================================
13.5023067523332

Trial 12 =========================================
13.960704773493044

Trial 13 =========================================
14.922677705491726

Trial 14 =============

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


Trial 15 =========================================
16.116007577142206

Trial 16 =========================================
15.255033568845326

Trial 17 =========================================
14.01321092442162

Trial 18 =========================================
15.09376663567193

Trial 19 =========================================
14.160452889708305

Trial 20 =========================================
14.802838786837714

Trial 21 =========================================
15.057143136506944

Trial 22 =========================================
14.331424736396789

Trial 23 =========================================
14.682619693026059

Trial 24 =========================================
16.604214665273517

Trial 25 =========================================
15.891416348273655

Trial 26 =========================================
13.764654668573245



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 27 =========================================
15.993635437267027

Trial 28 =========================================
13.975093660795855

Trial 29 =========================================
13.694484321716248

Trial 30 =========================================
14.967246540063524

Trial 31 =========================================
14.879874929833406

Trial 32 =========================================
14.583453767572223

Trial 33 =========================================
13.11549197562121

Trial 34 =========================================
14.807188519392271

Trial 35 =========================================
13.596317053680062

Trial 36 =========================================
15.236621223358435

Trial 37 =========================================
13.499385536085343

Trial 38 =========================================
14.49487549702145

Trial 39 =========================================
13.333217776457829

Trial 40 =========================================
15.167496801368138

Trial 41

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 47 =========================================
15.929222010399386

Trial 48 =========================================
14.216151580123185

Trial 49 =========================================
13.523545778028279

Trial 50 =========================================
15.644860315547838

Trial 51 =========================================
16.392306614028044

Trial 52 =========================================
13.918036162128976

Trial 53 =========================================
13.687905000131991

Trial 54 =========================================
14.821072002572953

Trial 55 =========================================
14.43273630695175

Trial 56 =========================================
13.647226545446836

Trial 57 =========================================
13.626065521563993

Trial 58 =========================================
13.931914501268144

Trial 59 =========================================
16.380945396393585

Trial 60 =========================================
12.888226429545892

Trial 6

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 17.93756973629349
Avg = 14.582782293414033
Std = 0.9587931759019597


In [6]:
print(y_max_arr.tolist())

[16.04736780513803, 14.37420797422643, 13.256687963205826, 17.93756973629349, 13.628919188978331, 13.651794540164985, 13.57460867368526, 16.038602500030457, 14.424137611898903, 14.148899262046438, 14.574528377016225, 13.5023067523332, 13.960704773493044, 14.922677705491726, 13.512981390238325, 16.116007577142206, 15.255033568845326, 14.01321092442162, 15.09376663567193, 14.160452889708305, 14.802838786837714, 15.057143136506944, 14.331424736396789, 14.682619693026059, 16.604214665273517, 15.891416348273655, 13.764654668573245, 15.993635437267027, 13.975093660795855, 13.694484321716248, 14.967246540063524, 14.879874929833406, 14.583453767572223, 13.11549197562121, 14.807188519392271, 13.596317053680062, 15.236621223358435, 13.499385536085343, 14.49487549702145, 13.333217776457829, 15.167496801368138, 13.670715701918313, 14.734248778202193, 13.126646598718555, 13.468375742691611, 14.927321057399322, 14.126083749699935, 15.929222010399386, 14.216151580123185, 13.523545778028279, 15.644860

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-24.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)